In [ ]:
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
import json
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template
from sentence_transformers import SentenceTransformer


In [ ]:
test_df = pd.read_csv('./Generative/progressive_test_20.csv')

msgs_df = pd.read_csv("./changed_processing_and_merged_with_hagai/arabic_messages.csv")
convs_df = pd.read_csv("./changed_processing_and_merged_with_hagai/arabic_conversations.csv")

In [ ]:
#########################
# 1. Load the Fine-Tuned Model
#########################
# IMPORTANT: Make sure you pass the local path you used to save your model
# e.g. new_model_local = "Gemma-3-12B-it-FirstResponder"
FINETUNED_MODEL_PATH = "./Generative/Gemma-3-12B-it-2000steps_Arapro_with_lexicon"

torch.cuda.empty_cache()

print("Loading the fine-tuned model...")
model, tokenizer = FastModel.from_pretrained(
    model_name = FINETUNED_MODEL_PATH,
    max_seq_length = 20000,
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
    # token = "hf_..."  # If your model is gated and requires a token
)

# Because we used the gemma-3 chat template, fetch it again
tokenizer = get_chat_template(
    tokenizer,
    chat_template="gemma-3",  # same template used during finetuning
)

model.eval()

In [ ]:
#########################
# 2. Load the Test Data
#########################
import pandas as pd

TEST_FILE = "./Generative/progressive_test_20.csv"

test_df = pd.read_csv(TEST_FILE)

# We expect columns: "input" and "output" (help-seeker input, first-responder reference)
test_df = test_df.dropna(subset=["input", "output"])  # just in case
gsr_ids = convs_df[convs_df.gsr==1]['engagement_id']
all_gsr = test_df[test_df.engagement_id.isin(gsr_ids.values)]
non_gsr = test_df[~test_df.engagement_id.isin(gsr_ids.values)]
print(all_gsr.shape)

test_df = pd.concat([non_gsr.sample(n=2800, random_state=42), all_gsr])

In [ ]:
import json, numpy as np
from sentence_transformers import SentenceTransformer

lexicon_df = pd.read_csv("./new_arabic_lexicon_17_07.csv")

lexicon_embeddings = np.load("./embeddings.npy")
embedding_model = SentenceTransformer('google/embeddinggemma-300m',
                               token = "your_token")

needed_categories = ["Past suicidal history", "Family suicide history", "Suicidal ideation", "Hopelessness", "Deliberate self harm", "Perceived burdensomeness"]

In [ ]:
#########################
# 3. Define System Prompt
#########################
import random

system_prompt = (
    "أنت مساعد دعم نفسي متعاطف ومليء بالرحمة، يقدم دعمًا عاطفيًا من خلال محادثات نصية للأشخاص الذين يطلبون المساعدة. "
    "دورك هو الاستماع بإنصات، وتأكيد مشاعرهم، وتقديم الدعم العاطفي. "
    "شجعهم بلطف، واطرح أسئلة مفتوحة، ووجّه المستخدمين نحو استراتيجيات تأقلم إيجابية. "
    "تجنب تقديم تشخيصات طبية أو توصيات لعلاج طبي. "
    "إذا ذكر المستخدم أنه في ضائقة فورية أو أشار إلى إيذاء النفس، فاقترح بلطف التواصل مع مختص في الصحة النفسية أو الاتصال بخدمات الطوارئ. "
    "حافظ دائمًا على مساحة آمنة وغير حُكمية يمكن للمستخدمين المشاركة فيها بحرية."
)




def build_query(msg):
    """
    given msg (str) build a query to the embedding gemma model (which works in query-document style)
    """
    prompt = "اي من هذه الجمل هي الاقرب من ناحية المضمون الى"
    prompt += f" '{msg}'"
    return prompt
    
def get_top_n_similar(msg, n=10):
    """
    given msg (str) get top n most similar phrases from the lexicon, then return their indices in the lexicon csv file
    to extract their categories later.
    output: list of integers (index of each phrase of the top n in the lexicon file)
    """
    global lexicon_embeddings
    query = build_query(msg)
    queries_embeddings = embedding_model.encode_query(query)
    similarities = pd.Series(embedding_model.similarity(queries_embeddings, lexicon_embeddings).flatten())
    return similarities.nlargest(n).index.tolist()


def categories_injection(user_input):
    """
    given user_input (str) get top 5 most similar phrases in the lexicon then extract their categories, and insert these categories
    into <context> tag to add it to the prompt later

    output: str containting <context> tag and catefories inside it. Example:  
        <context>
        Categories: Family suicide history, Preparatory acts, Family suicide history, Past suicidal history, Family suicide history
        </context>
    """
    top_5_categories = get_top_n_similar(user_input, 5)
    top_5_categories = lexicon_df.iloc[top_5_categories]['Category']
    
    formatted_cats = ", ".join(top_5_categories)
    categories_context = f"<context>\nCategories: {formatted_cats}\n</context>\n\n"
    return categories_context




def build_prompt(user_input: str):

    categories_as_context = categories_injection(user_input) # add domain knowledge to the prompt
    
    messages = [
        {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
        {"role": "user",   "content": [{"type": "text", "text": categories_as_context + user_input}]}
    ]
    
    return tokenizer.apply_chat_template(messages, add_generation_prompt=True)


    

print(build_prompt("بدي انتحر"))

<bos><start_of_turn>user
أنت مساعد دعم نفسي متعاطف ومليء بالرحمة، يقدم دعمًا عاطفيًا من خلال محادثات نصية للأشخاص الذين يطلبون المساعدة. دورك هو الاستماع بإنصات، وتأكيد مشاعرهم، وتقديم الدعم العاطفي. شجعهم بلطف، واطرح أسئلة مفتوحة، ووجّه المستخدمين نحو استراتيجيات تأقلم إيجابية. تجنب تقديم تشخيصات طبية أو توصيات لعلاج طبي. إذا ذكر المستخدم أنه في ضائقة فورية أو أشار إلى إيذاء النفس، فاقترح بلطف التواصل مع مختص في الصحة النفسية أو الاتصال بخدمات الطوارئ. حافظ دائمًا على مساحة آمنة وغير حُكمية يمكن للمستخدمين المشاركة فيها بحرية.

<context>
Categories: Family suicide history, Preparatory acts, Family suicide history, Past suicidal history, Family suicide history
</context>

بدي انتحر<end_of_turn>
<start_of_turn>model



In [ ]:
import json
#########################
# 4. Generate Predictions and Save
#########################
output_data = []

print("Running inference on test set...")
curr_idx = 0

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    if curr_idx % 2 == 0:
        with open(f'./Gemma-3-12B-it-2000steps_Arapro_with_lexicon/RAG_all_categories_Arapro_test_output.json', 'w') as json_file:
            json.dump([{"steps": curr_idx}]+output_data, json_file, indent=4)
    curr_idx += 1
    user_text = row["input"]
    reference_text = row["output"]
    
    prompt_text = build_prompt(user_text)

    inputs = tokenizer(text=prompt_text, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=1.0,
            top_p=0.95,
            top_k=64,
            do_sample=True
        )
    
    gen_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    output_data.append({
        "input": user_text,
        "reference": reference_text,
        "prediction": gen_text
    })


In [ ]:
# generate the same json file but for each prediction split by \nmodel\n
output_data_split = []
for data in output_data:
    user_text = data["input"]
    reference_text = data["reference"]
    prediction_text = data["prediction"].split("\nmodel\n")[-1]
    
    output_data_split.append({
        "input": user_text,
        "reference": reference_text,
        "prediction": prediction_text
    })
# Save to JSON
with open("./Gemma-3-12B-it-2000steps_Arapro_with_lexicon/RAG_all_categories_Arapro_test_output.json", "w", encoding="utf-8") as f:
    json.dump(output_data_split, f, ensure_ascii=False, indent=2)

Saved outputs to generation_outputs.json ✅
